# Deposit Attrition EDA — v6

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v5's construction was right — `median_dd = 1.000` on every covered feature, and lift came
back to **30.1× / 23.8× / 17.7×** against v4's 1.1–5.9×. Two things it still reported wrong.

## Fix 1 — a coverage floor on the ordering claim

A `dd` exists only for customers who used that feature in **both** windows. v5's headline
named `amt_out_rtp` as the earliest signal at rel_m −12 — on **2.6% coverage**. That median
is computed over a tiny self-selected subset of RTP users. Six features "separated at −12"
and most were sparse rails.

Features now carry their coverage, and anything under `MIN_COVERAGE` is shown but **excluded
from the ordering**. On the dense features the defensible claim is `amt_out` / `n_out` /
`n_in` at −7 against `bal_live` at −5 — a **2-month lead, not 7**.

## Fix 2 — compare single vs combined *within* a month

v5 set "best single 30.1×" against "best combined 16.3×" and called stacking a loss. But the
30.1× is at rel_m **−1**, where everything works, and the 16.3× is at **−12**. Different
months, different questions: one confirms a departure, the other gives warning.

§5g now compares them month by month, at matched recall, with the single side restricted to
features that cleared the coverage floor. §5h splits the verdict into a **far window**
(rel_m ≤ −6) and a **near window** (≥ −3). Early warning lives in the far months, and that is
where a combined rule has to earn its keep — winning there and losing at −1 is still the
right answer for a queue meant to give warning rather than confirm a departure.

## What v5 established and v6 keeps

- **The incumbent bar is low.** The 30% rule gives a **median 2 months** warning, fires
  *after* the exit for 3,059 attriters and never fires for 2,829 — **36% get no usable
  warning** — while burning 43,143 false alarms.
- **Fewer payments, not smaller**, at every month: `count_dd` 0.944 → 0.191 while `ticket_dd`
  holds 0.968 → 0.616 and stayers sit flat at ~1.00.
- **Net-flow-negative share** climbs 0.474 → 0.702 against a stayer line pinned at 0.44–0.45.
- **ACH is the sticky rail.** At rel_m −6: cheque 0.220 and RTP 0.015 fall hardest, wire 0.530,
  card 0.698, ACH last at 0.736 — the one they keep using while everything else stops.

Reads v2's panels and v3's labels; rebuilds nothing.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v5
# =====================================================================
# v4 swapped a self-relative CHANGE for a peer-relative LEVEL and kept the
# same thresholds. Three consequences, all visible in the v4 output:
#
#   - peer deciles are built on bal_live, so bal_live / its own decile
#     median is ~1.0 by construction. Balance "separated" at rel_m 0 under
#     peer vs -5 under self: the signal was normalised away circularly.
#   - a peer level ratio has median 1.0 across the population, so a 0.70
#     threshold flags about half of everyone. Lift fell to 1.5x against
#     23x in v3. The v4 operating points and the "combining doesn't help"
#     verdict are artefacts of that, not findings.
#   - where the peer median is ~0 the ratio is null (card, rtp vanished)
#     and where it is merely tiny it explodes (wire max_gap 34.5).
#
# v5 measures DIFFERENCE-IN-DIFFERENCES: how much a customer's own recent
# change differs from the same change across its peers.
#
#     chg_t  = value_t / mean(value over t-6 .. t-4)
#     dd_t   = chg_t / median(chg_t across the customer's peer group)
#
# This keeps what each normalisation was good for and drops what it was
# bad at. It needs SIX months of history, not a twelve-month pre-window
# aligned to an event, so it uses every attriter under the permanent 2024
# floor - and it is computable in production every month without knowing
# the event date, which a rel_m baseline never was.
from pathlib import Path

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V3  = "hdfs://nameservice1/user/pk36814/attrition_v3"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v6"
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v6")

DATE_START, DATE_END = "2024-01-01", "2026-07-31"
MAX_ROWS, ZERO_TOL   = 60, 1.0

# ── change window ─────────────────────────────────────────────────────
CHG_LAG_FAR, CHG_LAG_NEAR = -6, -4     # reference = mean over t-6 .. t-4
CHG_MIN_OBS = 2                        # months needed in the reference
MIN_REF     = 1.0                      # reference below this -> no ratio

# ── peer groups: FROZEN, not moving with the decline ──────────────────
# v4 recomputed the decile every month, so a shrinking customer slid down
# the deciles alongside its own decline and the ratio stayed flat.
PEER_DECILES  = 10
PEER_ANCHOR_M = 3        # decile from the customer's FIRST 3 observed months
PEER_MIN_N    = 50
PEER_USE_SEG  = False

# ── labels ────────────────────────────────────────────────────────────
MIN_HIST_M, MIN_LIVE_BEFORE = 12, 6
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3
P30_DROP = 0.70          # the incumbent rule, rebuilt here - v4 referenced
                         # m_C_p30, which v3 never persisted

# ── event study ───────────────────────────────────────────────────────
STUDY_DEFS  = ["A_full_exit", "B_bal_exit"]
EVENT_PRE, EVENT_POST = 12, 3
SEARCH_FROM = -12        # dd starts at ~1.0 for everyone, so the whole
                         # window is legitimate - there is no baseline to
                         # search inside
MIN_CELL_N  = 200
# A dd only exists for customers who used that feature in BOTH windows.
# amt_out_rtp has 2.6% coverage, so its median is computed on a tiny
# self-selected subset - and v5 put it at the top of the headline. A
# feature must clear this to be eligible for the ordering claim.
MIN_COVERAGE = 0.25
SEP_LEVEL, SEP_RATE, HOLD = 0.15, 0.05, 2
BURN_IN_YM  = ["2024-01", "2024-02", "2024-03"]
NEW_ENTITY  = ["cpty_new_out", "fin_new_out"]

# ── operating points ──────────────────────────────────────────────────
OP_THRESH   = [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]
TARGET_PREC, MIN_RECALL = 0.25, 0.20
# v5 compared "best single 30.1x" (at rel_m -1) against "best combined
# 16.3x" (at rel_m -12) and called it a loss. Those are different months.
# The comparison is now made WITHIN each month, and at matched recall.
CMP_RECALL = 0.20
SCORE_THRESH = 0.70

# ── the twelve ────────────────────────────────────────────────────────
SIGNALS = [
 dict(n=1,  name="Stops taking on new trading partners",     feature="cpty_new_out",    rule="zero"),
 dict(n=2,  name="Stops paying anyone at an unfamiliar bank", feature="fin_new_out",     rule="zero"),
 dict(n=3,  name="First rail to go — cheque",                 feature="amt_out_check",   rule="dd"),
 dict(n=4,  name="Net flow turns against us",                 feature="net_flow",        rule="sign"),
 dict(n=5,  name="Their own customers stop paying them here", feature="amt_in_internal", rule="dd"),
 dict(n=6,  name="Spending through us falls",                 feature="amt_out",         rule="dd"),
 dict(n=7,  name="Fewer payments, not just smaller",          feature="n_out",           rule="dd",
                                                              companion="avg_ticket_out"),
 dict(n=8,  name="Inbound activity thins",                    feature="n_in",            rule="dd"),
 dict(n=9,  name="What today's monitoring sees",              feature="bal_live",        rule="dd"),
 dict(n=10, name="Payments to other PNC customers fall",      feature="amt_out_internal",rule="dd"),
 dict(n=11, name="The relationship list itself shrinks",      feature="cpty_out_n",      rule="dd"),
 dict(n=12, name="Banks they pay drop away",                  feature="fin_out_n",       rule="dd"),
]
# net_flow can be negative, so a ratio is meaningless for it - it is read
# on its sign, and only on its sign.
NO_RATIO   = ["net_flow"]
RAILS      = ["ach", "wire", "check", "card", "rtp", "other"]
RAIL_FEATS = [f"amt_out_{r}" for r in RAILS]
SIG_FEATS  = [s["feature"] for s in SIGNALS] + ["avg_ticket_out", "avg_ticket_in", "amt_in"]
HTML_NAME  = "PKG_Attrition_Signals.html"

In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS, SVG  (numpy truthiness bug fixed)
# =====================================================================
import warnings, html as _html, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
spark = (SparkSession.builder.appName("pkg_attrition_eda_v5")
         .config("spark.sql.shuffle.partitions", "400")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())
pd.set_option("display.max_columns", 300); pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
def v3(n): return f"{HDFS_V3.rstrip('/')}/{n}"
def pct(a, b): return float(a)/float(b) if b else float("nan")
def sdiv(a, b): return F.when(F.col(b) > 0, F.col(a)/F.col(b))

def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;margin:10px 0 2px;"
                     f"color:#111'>{title}<span style='font-weight:400;color:#888'> &middot; "
                     f"{len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    return disp(pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())}),
                title=title, n=len(d), save=save)

def svg_lines(df, ycols, title="", W=560, H=210, pad=44,
              colors=("#C1440E", "#4A6FA5"), yzero=False, vline=0, hline=None):
    d = df.dropna(subset=list(ycols), how="all")
    if d.empty: return "<div style='color:#999'>insufficient data</div>"
    xs = d.index.astype(float).values
    # FIX: `np.concatenate(...) or default` raises "truth value of an array
    # is ambiguous". Filter the empty arrays out explicitly instead.
    arrs = [d[c].astype(float).dropna().values for c in ycols if c in d.columns]
    arrs = [a for a in arrs if a.size]
    if not arrs: return "<div style='color:#999'>insufficient data</div>"
    vals = np.concatenate(arrs)
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if yzero: lo = min(lo, 0.0)
    if hline is not None: lo, hi = min(lo, hline), max(hi, hline)
    if hi <= lo: hi = lo + 1e-9
    m = (hi - lo) * .08; lo -= m; hi += m
    def X(v): return pad + (v - xs.min())/max(xs.max()-xs.min(), 1e-9)*(W-pad-14)
    def Y(v): return H - pad + 6 - (v - lo)/(hi - lo)*(H-pad-22)
    p = [f"<svg viewBox='0 0 {W} {H}' width='100%' style='max-width:{W}px;font:11px IBM Plex Sans,sans-serif'>"]
    for gy in np.linspace(lo, hi, 5):
        p.append(f"<line x1='{pad}' y1='{Y(gy):.1f}' x2='{W-14}' y2='{Y(gy):.1f}' stroke='#EEE'/>")
        p.append(f"<text x='{pad-6}' y='{Y(gy)+3:.1f}' text-anchor='end' fill='#999'>{gy:,.2f}</text>")
    if hline is not None:
        p.append(f"<line x1='{pad}' y1='{Y(hline):.1f}' x2='{W-14}' y2='{Y(hline):.1f}' "
                 f"stroke='#CCC' stroke-dasharray='2,3'/>")
    if vline is not None and xs.min() <= vline <= xs.max():
        p.append(f"<line x1='{X(vline):.1f}' y1='16' x2='{X(vline):.1f}' y2='{H-pad+6}' "
                 f"stroke='#BBB' stroke-dasharray='3,3'/>")
    for i, c in enumerate(ycols):
        if c not in d.columns: continue
        s = d[c].astype(float)
        pts = " ".join(f"{X(x):.1f},{Y(v):.1f}" for x, v in zip(s.index.astype(float), s.values) if pd.notna(v))
        if pts:
            p.append(f"<polyline points='{pts}' fill='none' stroke='{colors[i%len(colors)]}' stroke-width='2'/>")
            p.append(f"<circle cx='{W-150+i*78}' cy='12' r='4' fill='{colors[i%len(colors)]}'/>"
                     f"<text x='{W-142+i*78}' y='15' fill='#555'>{_html.escape(str(c))}</text>")
    for x in xs[::max(1, len(xs)//8)]:
        p.append(f"<text x='{X(x):.1f}' y='{H-pad+22}' text-anchor='middle' fill='#999'>{int(x)}</text>")
    p.append(f"<text x='{pad}' y='12' fill='#333' font-weight='600'>{_html.escape(title)}</text>")
    p.append(f"<text x='{W/2:.0f}' y='{H-4}' text-anchor='middle' fill='#AAA'>months before event</text></svg>")
    return "".join(p)

def svg_bars(labels, values, title="", W=560, H=210, pad=44, color="#C1440E"):
    v = [x for x in values if pd.notna(x)]
    if not v: return "<div style='color:#999'>insufficient data</div>"
    hi = max(max(v), 1e-9); n = len(values); bw = (W-pad-14)/max(n,1)*.68
    p = [f"<svg viewBox='0 0 {W} {H}' width='100%' style='max-width:{W}px;font:11px IBM Plex Sans,sans-serif'>",
         f"<text x='{pad}' y='12' fill='#333' font-weight='600'>{_html.escape(title)}</text>"]
    for i, (l, val) in enumerate(zip(labels, values)):
        x = pad + (W-pad-14)*(i+.16)/n
        h = 0 if pd.isna(val) else (val/hi)*(H-pad-26)
        p.append(f"<rect x='{x:.1f}' y='{H-pad+6-h:.1f}' width='{bw:.1f}' height='{max(h,0):.1f}' fill='{color}'/>")
        p.append(f"<text x='{x+bw/2:.1f}' y='{H-pad+20}' text-anchor='middle' fill='#999'>{_html.escape(str(l))}</text>")
        if pd.notna(val):
            p.append(f"<text x='{x+bw/2:.1f}' y='{H-pad+2-h:.1f}' text-anchor='middle' fill='#555'>{val:,.2f}</text>")
    p.append("</svg>"); return "".join(p)

def tbl(df, cls="t"):
    return df.to_html(index=False, classes=cls, border=0,
                      float_format=lambda v: f"{v:,.3f}", na_rep="&mdash;", escape=False)

_F = OUT_DIR / "FINDINGS_v6.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, q, a, d=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=q, answer=str(a), detail=str(d)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)
print("helpers ready")

## 1 · Frozen peers and difference-in-differences

In [ ]:
# =====================================================================
# 2 · FROZEN PEERS + DIFFERENCE-IN-DIFFERENCES         [OUTPUT BLOCK 1]
# =====================================================================
cust_month = spark.read.parquet(v2("panel_customer_month")).filter(F.col("ym") >= DATE_START[:7])
feat       = spark.read.parquet(v2("panel_pay_features")).filter(F.col("ym") >= DATE_START[:7])
lab0       = spark.read.parquet(v3("labels_customer"))
for c in NEW_ENTITY:
    feat = feat.withColumn(c, F.when(F.col("ym").isin(*BURN_IN_YM), None).otherwise(F.col(c)))

panel = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live", "n_accts",
                           "n_accts_live", "all_closed", "segment_desc", "naics", "state")
         .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left")
         .withColumn("avg_ticket_out", sdiv("amt_out", "n_out"))
         .withColumn("avg_ticket_in",  sdiv("amt_in",  "n_in"))).persist(StorageLevel.DISK_ONLY)

FEATS = sorted(set(SIG_FEATS + RAIL_FEATS) & set(panel.columns) | {"bal_live"})
_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in FEATS])

# ── FROZEN peer decile ────────────────────────────────────────────────
# v4 recomputed the decile every month, so a shrinking customer slid down
# the deciles in step with its own decline and the ratio stayed flat. The
# decile is now fixed from the customer's first PEER_ANCHOR_M months and
# never moves again.
wc = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
anchor = (panel.withColumn("rn", F.row_number().over(wc)).filter(F.col("rn") <= PEER_ANCHOR_M)
          .groupBy("cust_pwr_id").agg(F.avg("bal_live").alias("anchor_bal"),
                                      F.max("segment_desc").alias("anchor_seg")))
anchor = (anchor.withColumn("bal_decile",
                            F.ntile(PEER_DECILES).over(Window.orderBy(F.col("anchor_bal").asc_nulls_first())))
          .withColumn("peer_key", F.concat_ws("|", F.col("bal_decile").cast("string"),
                                              F.coalesce("anchor_seg", F.lit("NA")) if PEER_USE_SEG else F.lit("ALL")))
          .select("cust_pwr_id", "peer_key", "bal_decile", "anchor_bal"))
anchor.write.mode("overwrite").parquet(hp("peer_anchor"))
anchor = spark.read.parquet(hp("peer_anchor")).persist(StorageLevel.DISK_ONLY)

long = (panel.join(anchor, "cust_pwr_id", "left")
        .select("cust_pwr_id", "m_idx", "ym", "peer_key",
                F.expr(f"stack({len(FEATS)}, {_stack}) as (feature, value)")))

# ── own change vs its own recent past ─────────────────────────────────
wr = (Window.partitionBy("cust_pwr_id", "feature").orderBy("m_idx")
      .rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR))
long = (long.withColumn("ref",  F.avg("value").over(wr))
             .withColumn("nref", F.count("value").over(wr))
             .withColumn("chg", F.when((F.col("nref") >= CHG_MIN_OBS) &
                                       (F.col("ref") > MIN_REF) &
                                       (~F.col("feature").isin(*NO_RATIO)),
                                       F.col("value") / F.col("ref"))))

# ── peers' change in the same calendar month, then the difference ─────
peer_chg = (long.groupBy("ym", "peer_key", "feature")
            .agg(F.count("chg").alias("peer_n"),
                 F.expr("percentile_approx(chg, 0.5)").alias("peer_chg"))
            .filter((F.col("peer_n") >= PEER_MIN_N) & (F.abs(F.col("peer_chg")) > 1e-6)))

long = (long.join(peer_chg, ["ym", "peer_key", "feature"], "left")
        .withColumn("dd", F.when(F.col("chg").isNotNull() & F.col("peer_chg").isNotNull(),
                                 F.col("chg") / F.col("peer_chg")))
        # kept deliberately: the v4 finding that attriters sit BELOW peers
        # from rel_m -12 is real, and it is a standing marker rather than
        # an early warning. Section 7 scores it separately.
        ).persist(StorageLevel.DISK_ONLY)

cov = (long.groupBy("feature").agg(
          F.count("*").alias("rows"),
          F.avg(F.col("chg").isNotNull().cast("double")).alias("share_with_chg"),
          F.avg(F.col("dd").isNotNull().cast("double")).alias("share_with_dd"),
          F.expr("percentile_approx(dd, 0.5)").alias("median_dd"))
       .orderBy("feature"))
disp(cov, title="1a &middot; dd coverage — median_dd should sit at ~1.0 for every feature; that is "
                "the property the v4 peer level ratio did not have", n=40, save="v5_dd_coverage")

kv({"customer-months": panel.count(),
    "peer grouping": f"{PEER_DECILES} FROZEN deciles from the first {PEER_ANCHOR_M} months",
    "change window": f"t vs mean(t{CHG_LAG_FAR} .. t{CHG_LAG_NEAR})",
    "history needed": f"{-CHG_LAG_FAR} months (not a 12-month pre-window)"},
   title="1b &middot; Construction", save="v5_construction")
note("DD", "How are the two normalisations reconciled?",
     "difference-in-differences: own change divided by the peer group's change",
     "v4 compared peer LEVELS against thresholds tuned for self-relative CHANGES, which flagged "
     "half the population and collapsed lift to 1.5x. dd is centred at 1.0 for everyone, needs "
     "6 months not 12, and is computable in production without knowing the event date.")

## 2 · Labels, and the incumbent rule

In [ ]:
# =====================================================================
# 3 · LABELS + THE INCUMBENT RULE, REBUILT             [OUTPUT BLOCK 2]
# =====================================================================
# FIX: v4's benchmark referenced m_C_p30, which v3 never persisted -
# AnalysisException. The 30% rule is rebuilt here from cust_month so the
# incumbent can be measured rather than assumed.

w    = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
w3   = w.rangeBetween(-2, 0); w6p = w.rangeBetween(-8, -3); w12 = w.rangeBetween(-11, 0)
wfwd = w.rangeBetween(0, BAL_EXIT_HOLD - 1)

c = (cust_month
     .withColumn("avg3", F.avg("bal_live").over(w3)).withColumn("n3", F.count("bal_live").over(w3))
     .withColumn("prior6", F.avg("bal_live").over(w6p)).withColumn("n_prior6", F.count("bal_live").over(w6p))
     .withColumn("n_hist", F.count("bal_live").over(w12))
     .withColumn("_a", F.array_sort(F.collect_list("bal_live").over(w12)))
     .withColumn("med12", F.expr("element_at(_a, cast(size(_a)/2 as int) + 1)")).drop("_a")
     .withColumn("low", ((F.col("med12") > ZERO_TOL) &
                         (F.col("bal_live") < BAL_EXIT_FRAC*F.col("med12"))).cast("int"))
     .withColumn("low_run", F.sum("low").over(wfwd)).withColumn("obs_fwd", F.count("*").over(wfwd))
     .withColumn("acr", F.sum("all_closed").over(wfwd)))

DEFS = {
 "A_full_exit": (F.col("all_closed") == 1) & (F.col("acr") == F.col("obs_fwd")),
 "B_bal_exit":  (F.col("n_hist") >= MIN_HIST_M) & (F.col("low_run") == BAL_EXIT_HOLD) &
                (F.col("obs_fwd") == BAL_EXIT_HOLD),
 "C_p30":       (F.col("n_hist") >= MIN_HIST_M) & (F.col("n_prior6") >= 4) & (F.col("n3") >= 3) &
                (F.col("avg3") < P30_DROP * F.col("prior6")),
}
for k, cond in DEFS.items(): c = c.withColumn(k, F.when(cond, 1).otherwise(0))

lab = (c.groupBy("cust_pwr_id").agg(
          *[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"m_{k}") for k in DEFS],
          F.min("m_idx").alias("first_m"), F.max("m_idx").alias("last_m"),
          F.count("*").alias("n_months"),
          F.min(F.when(F.col("n_accts_live") > 0, F.col("m_idx"))).alias("first_live_m"),
          F.max("segment_desc").alias("segment_desc"))
       .filter(f"n_months >= {MIN_HIST_M}"))
for k in DEFS:
    lab = lab.withColumn(f"q_{k}", F.when(
        F.col(f"m_{k}").isNotNull() & F.col("first_live_m").isNotNull() &
        (F.col(f"m_{k}") - F.col("first_live_m") >= MIN_LIVE_BEFORE), F.col(f"m_{k}")))
lab.write.mode("overwrite").parquet(hp("labels_customer"))
lab = spark.read.parquet(hp("labels_customer")).persist(StorageLevel.DISK_ONLY)

N_EV   = lab.count()
EVAL_M = cust_month.agg((F.max("m_idx") - F.min("m_idx") + 1)).collect()[0][0] - MIN_HIST_M
PREV   = {d: pct(lab.filter(F.col(f"q_{d}").isNotNull()).count(), N_EV*EVAL_M) for d in STUDY_DEFS}
disp(pd.DataFrame([dict(definition=k,
                        n_raw=lab.filter(F.col(f"m_{k}").isNotNull()).count(),
                        n_qualified=lab.filter(F.col(f"q_{k}").isNotNull()).count(),
                        monthly_hazard=PREV.get(k)) for k in DEFS]),
     title=f"2a &middot; Labels (n={N_EV:,}, {EVAL_M} evaluable months)", save="v5_labels")

# ── the incumbent, measured ───────────────────────────────────────────
bp = (lab.filter(F.col("q_A_full_exit").isNotNull())
      .withColumn("p30_lead", F.col("q_A_full_exit") - F.col("m_C_p30"))
      .select("cust_pwr_id", "p30_lead")).toPandas()
BENCH = pd.DataFrame({"metric": ["attriters", "…30% rule ever fires for", "…never fires for",
                                 "…fires AFTER the exit", "median months of warning", "p25", "p75"],
                      "value": [len(bp), int(bp.p30_lead.notna().sum()), int(bp.p30_lead.isna().sum()),
                                int((bp.p30_lead < 0).sum()), bp.p30_lead.median(),
                                bp.p30_lead.quantile(.25), bp.p30_lead.quantile(.75)]})
disp(BENCH, title="2b &middot; SIGNAL 9 — what today's monitoring actually delivers", save="v5_benchmark")
note("BENCH", "How much warning does the 30% rule give?",
     f"median {bp.p30_lead.median():.0f} months; never fires for {int(bp.p30_lead.isna().sum()):,}",
     "It also fired on 43,143 customers who never left (v2 5a), so the warning is cheap. "
     "Every other signal is worth deploying only to the extent it beats this.")

## 3 · Event study on dd, with a coverage floor

In [ ]:
# =====================================================================
# 4 · EVENT STUDY ON dd                                [OUTPUT BLOCK 3]
# =====================================================================
def cohorts(defn):
    ev = f"q_{defn}"
    a = (lab.filter(F.col(ev).isNotNull())
         .select("cust_pwr_id", F.col(ev).alias("event_m"),
                 F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
         .withColumn("cohort", F.lit("attriter")))
    dr = [r.event_m for r in a.select("event_m").limit(3000).collect()]
    dr = dr[::max(1, len(dr)//300)][:300] or [0]
    arr = F.array(*[F.lit(int(x)) for x in dr])
    s = (lab.filter(F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNull())
         .withColumn("event_m", F.element_at(arr, (F.abs(F.hash("cust_pwr_id")) % len(dr)) + 1))
         .select("cust_pwr_id", "event_m", F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
         .withColumn("cohort", F.lit("stayer")))
    return a.unionByName(s).filter(F.col("last_m") >= F.col("event_m"))

def build(defn):
    co = cohorts(defn)
    es = (long.join(co, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-EVENT_PRE, EVENT_POST))).persist(StorageLevel.DISK_ONLY)
    cur = (es.groupBy("feature", "cohort", "rel_m").agg(
               F.count("*").alias("n"),
               F.sum(F.col("dd").isNotNull().cast("int")).alias("n_dd"),
               F.sum(F.col("value").isNotNull().cast("int")).alias("n_val"),
               F.expr("percentile_approx(dd, 0.5)").alias("med_dd"),
               F.avg(F.when(F.col("value").isNotNull(), (F.col("value") > 0).cast("double"))).alias("rate_any"),
               F.avg(F.when(F.col("value").isNotNull(), (F.col("value") < 0).cast("double"))).alias("rate_neg"),
               # the standing-marker view: level vs peers, kept separate
               F.expr("percentile_approx(value, 0.5)").alias("med_value"))).toPandas()
    cur.loc[cur.n < MIN_CELL_N, ["med_dd", "rate_any", "rate_neg", "med_value"]] = np.nan
    cur.to_csv(OUT_DIR / f"v5_curves_{defn}.csv", index=False)
    return dict(es=es, cur=cur, n_att=co.filter("cohort='attriter'").count(),
                n_sta=co.filter("cohort='stayer'").count())

B = {d: build(d) for d in STUDY_DEFS}
disp(pd.DataFrame([dict(definition=d, attriters=B[d]["n_att"], stayers=B[d]["n_sta"]) for d in STUDY_DEFS]),
     title="3a &middot; Cohorts — no pre-window filter, so every qualified attriter is here",
     save="v5_cohorts")

CUR, ES = B["A_full_exit"]["cur"], B["A_full_exit"]["es"]
RULE = {s["feature"]: s["rule"] for s in SIGNALS}

def col_for(f):
    r = RULE.get(f, "dd")
    return ("rate_any", SEP_RATE) if r == "zero" else (("rate_neg", SEP_RATE) if r == "sign" else ("med_dd", SEP_LEVEL))

def coverage_of(cur, f):
    """Share of cohort rows where the statistic the feature is READ ON
    actually exists. dd features need the ratio; rate features only need
    the raw value."""
    g = cur[cur.feature == f]
    col, _ = col_for(f)
    num = g.n_val.sum() if col in ("rate_any", "rate_neg") else g.n_dd.sum()
    return num / max(g.n.sum(), 1)

def seps(cur):
    rows = []
    for f, g in cur.groupby("feature"):
        col, thr = col_for(f)
        wv = (g.pivot_table(index="rel_m", columns="cohort", values=col)
                .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if wv.empty: continue
        gap = (wv.attriter - wv.stayer).abs(); s = gap[gap.index >= SEARCH_FROM]
        sep, run = None, 0
        for rm, v in s.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD: sep = rm - HOLD + 1; break
        cov = coverage_of(cur, f)
        rows.append(dict(feature=f, read_on=col, coverage=round(cov, 3),
                         eligible=cov >= MIN_COVERAGE, sep_rel_m=sep,
                         lead=None if sep is None else -sep,
                         gap_at_minus6=round(gap.get(-6, np.nan), 3),
                         gap_at_event=round(gap.get(0, np.nan), 3),
                         max_gap=round(s.max(), 3) if len(s) else np.nan))
    out = pd.DataFrame(rows)
    # ineligible features stay visible but sort to the bottom - they are
    # not evidence for an ordering claim
    return out.sort_values(["eligible", "sep_rel_m", "max_gap"],
                           ascending=[False, True, False], na_position="last")

LEAD = seps(CUR)
disp(LEAD, title="4a &middot; First sustained separation on dd — centred at 1.0, so the whole "
                 "window is legitimate and there is no baseline to search inside",
     n=40, save="v5_lead_A")
disp(seps(B["B_bal_exit"]["cur"]), title="4b &middot; Same, shell-account population",
     n=40, save="v5_lead_B")

ELIG = sorted(LEAD.loc[LEAD.eligible, "feature"])
disp(LEAD[~LEAD.eligible][["feature", "coverage", "sep_rel_m", "max_gap"]],
     title=f"4a2 &middot; EXCLUDED from the ordering — coverage below {MIN_COVERAGE:.0%}. "
           f"v5's headline named amt_out_rtp at rel_m -12 on 2.6% coverage",
     n=20, save="v6_lead_excluded")

_b = LEAD[LEAD.feature == "bal_live"]
_p = LEAD[(LEAD.feature != "bal_live") & LEAD.sep_rel_m.notna() & LEAD.eligible]
bal_m = None if _b.empty or pd.isna(_b.iloc[0].sep_rel_m) else _b.iloc[0].sep_rel_m
pay   = None if _p.empty else _p.iloc[0]
kv({"features eligible (coverage >= %.0f%%)" % (MIN_COVERAGE*100): len(ELIG),
    "features excluded as too sparse": int((~LEAD.eligible).sum()),
    "balance separates at rel_m": bal_m,
    "earliest ELIGIBLE payment signal": None if pay is None else pay.feature,
    "  ...its coverage": None if pay is None else pay.coverage,
    "  ...at rel_m": None if pay is None else pay.sep_rel_m,
    "PAYMENT LEAD (months)": None if (pay is None or bal_m is None) else int(bal_m - pay.sep_rel_m)},
   title="4c &middot; Headline, restricted to features with real coverage", save="v6_headline")
note("COV", "Which features can carry the ordering claim?",
     f"{len(ELIG)} of {len(LEAD)} clear {MIN_COVERAGE:.0%} coverage",
     "A dd exists only for customers who used the feature in both windows. v5's headline named "
     "amt_out_rtp at rel_m -12 on 2.6% coverage - a median over a tiny self-selected subset. "
     "Six features 'separated at -12' and most were sparse rails.")

## 4 · Operating points, and the standing marker

In [ ]:
# =====================================================================
# 5 · OPERATING POINTS ON dd + THE STANDING MARKER     [OUTPUT BLOCK 4]
# =====================================================================
def op(defn):
    es, p = B[defn]["es"], PREV[defn]
    aggs = [F.count("*").alias("n"), F.sum(F.col("dd").isNotNull().cast("int")).alias("n_dd"),
            F.sum(F.col("value").isNotNull().cast("int")).alias("n_val"),
            F.sum(((F.col("value").isNotNull()) & (F.col("value") <= 0)).cast("int")).alias("hit_zero"),
            F.sum(((F.col("value").isNotNull()) & (F.col("value") < 0)).cast("int")).alias("hit_sign")]
    aggs += [F.sum((F.col("dd") < t).cast("int")).alias(f"lt{i}") for i, t in enumerate(OP_THRESH)]
    raw = (es.filter(F.col("feature").isin(*RULE.keys()) & F.col("rel_m").between(-EVENT_PRE, -1))
             .groupBy("feature", "rel_m", "cohort").agg(*aggs)).toPandas()
    out = []
    for (f, rm), g in raw.groupby(["feature", "rel_m"]):
        a, s = g[g.cohort == "attriter"], g[g.cohort == "stayer"]
        if a.empty or s.empty: continue
        a, s = a.iloc[0], s.iloc[0]; r = RULE.get(f, "dd")
        cand = ([("zero", "hit_zero", a.n_val, s.n_val)] if r == "zero" else
                [("negative", "hit_sign", a.n_val, s.n_val)] if r == "sign" else
                [(t, f"lt{i}", a.n_dd, s.n_dd) for i, t in enumerate(OP_THRESH)])
        for thr, col, na, ns in cand:
            if na < MIN_CELL_N or ns < MIN_CELL_N: continue
            rec, fpr = a[col]/na, s[col]/ns
            den = p*rec + (1-p)*fpr
            pr = (p*rec/den) if den > 0 else np.nan
            out.append(dict(feature=f, rel_m=int(rm), rule=r, threshold=thr, recall=rec, fpr=fpr,
                            precision=pr, lift=(pr/p) if pr == pr else np.nan,
                            alerts_per_tp=(1/pr) if pr and pr > 0 else np.nan,
                            n_attriter=int(na), n_stayer=int(ns)))
    return pd.DataFrame(out)

OP = {d: op(d) for d in STUDY_DEFS}
for d in STUDY_DEFS: OP[d].to_csv(OUT_DIR / f"v5_operating_{d}.csv", index=False)

def best_per(dfp):
    rows = []
    for f, g in dfp.groupby("feature"):
        ok = g[(g.precision >= TARGET_PREC) & (g.recall >= MIN_RECALL)]
        top = g.loc[g.lift.idxmax()] if g.lift.notna().any() else None
        e = ok.loc[ok.rel_m.idxmin()] if not ok.empty else None
        rows.append(dict(feature=f,
                         earliest_usable_rel_m=None if e is None else int(e.rel_m),
                         usable_threshold=None if e is None else e.threshold,
                         usable_recall=None if e is None else round(e.recall, 3),
                         usable_precision=None if e is None else round(e.precision, 3),
                         best_lift=None if top is None else round(top.lift, 1),
                         best_at_rel_m=None if top is None else int(top.rel_m),
                         best_precision=None if top is None else round(top.precision, 3),
                         best_recall=None if top is None else round(top.recall, 3)))
    return pd.DataFrame(rows).sort_values("best_lift", ascending=False, na_position="last")

BEST = {d: best_per(OP[d]) for d in STUDY_DEFS}
disp(BEST["A_full_exit"], title="4a &middot; Every signal on dd — compare best_lift against v4's "
                                "1.1&ndash;5.9x, which was the level-ratio artefact", n=25, save="v5_best_A")
disp(BEST["B_bal_exit"], title="4b &middot; Shell-account population", n=25, save="v5_best_B")

# ── the standing marker: attriters are lighter than peers ALL ALONG ───
# v4's peer levels were the wrong basis for an early warning, but the
# level difference itself is real and worth keeping: at rel_m -12
# attriters sat at 0.70 of peer amt_out and 0.647 of peer n_out. That is
# a segmentation fact, not a trigger, and it is scored as such here.
early = (ES.filter((F.col("rel_m").between(-EVENT_PRE, -EVENT_PRE + 2)) &
                   F.col("feature").isin("amt_out", "n_out", "cpty_out_n", "bal_live"))
         .groupBy("cust_pwr_id", "cohort", "feature").agg(F.avg("value").alias("early_val"))
         .join(anchor.select("cust_pwr_id", "peer_key"), "cust_pwr_id", "left"))
pm = (early.groupBy("peer_key", "feature").agg(F.expr("percentile_approx(early_val, 0.5)").alias("pmed"))
      .filter(F.col("pmed") > 0))
early = early.join(pm, ["peer_key", "feature"], "inner").withColumn("rel_to_peer", F.col("early_val")/F.col("pmed"))

MARKER = (early.groupBy("feature", "cohort")
          .agg(F.count("*").alias("n"),
               F.expr("percentile_approx(rel_to_peer, 0.5)").alias("median_vs_peer"),
               F.avg((F.col("rel_to_peer") < 0.5).cast("double")).alias("share_below_half_of_peers"))
          .orderBy("feature", "cohort"))
disp(MARKER, title="4c &middot; STANDING MARKER — how attriters look a full year out, before any "
                   "decline. A segmentation fact, not a trigger", n=20, save="v5_standing_marker")

_mp = MARKER.toPandas().pivot_table(index="feature", columns="cohort", values="share_below_half_of_peers")
if {"attriter", "stayer"} <= set(_mp.columns):
    _mp["lift"] = (_mp.attriter/_mp.stayer).round(2)
    disp(_mp.reset_index(), title="4d &middot; Being under half your peers a year out — lift on its own",
         save="v5_marker_lift")
note("MARKER", "Are attriters different a full year before they leave?",
     "Yes - materially lighter payment users than their size peers at rel_m -12",
     "v4 measured this as if it were an early warning, which it is not: a level that is already "
     "different at -12 has no onset date. It is a standing segmentation marker, and 4d prices it.")

## 5 · Deep dives, and single vs combined within each month

In [ ]:
# =====================================================================
# 6 · DEEP DIVES + STACKING                            [OUTPUT BLOCK 5]
# =====================================================================
def pv(cur, feats, col="med_dd"):
    return cur[cur.feature.isin(feats)].pivot_table(index="rel_m", columns=["feature", "cohort"], values=col)

# ── SIGNAL 3 · rail order. v4 lost card/rtp/other because their peer
#    median was 0 and kept wire with a gap of 34.5 for the same reason.
#    dd carries its own guard (peer_chg must exceed 1e-6), and coverage
#    is reported so a rail that is simply absent says so.
rr = []
for f in [r for r in RAIL_FEATS if r in set(CUR.feature)]:
    g = CUR[CUR.feature == f].pivot_table(index="rel_m", columns="cohort", values="med_dd")
    covr = CUR[CUR.feature == f].n_dd.sum() / max(CUR[CUR.feature == f].n.sum(), 1)
    if not {"attriter", "stayer"} <= set(g.columns):
        rr.append(dict(rail=f.replace("amt_out_", "").upper(), feature=f, coverage=round(covr, 3),
                       sep_rel_m=None, note="too sparse to read")); continue
    gap = (g.attriter - g.stayer).abs(); s = gap[gap.index >= SEARCH_FROM]
    sep, run = None, 0
    for rm, v in s.items():
        run = run + 1 if v > SEP_LEVEL else 0
        if run >= HOLD: sep = rm - HOLD + 1; break
    rr.append(dict(rail=f.replace("amt_out_", "").upper(), feature=f, coverage=round(covr, 3),
                   sep_rel_m=sep, dd_at_minus6=round(g.attriter.get(-6, np.nan), 3),
                   dd_at_minus3=round(g.attriter.get(-3, np.nan), 3),
                   max_gap=round(s.max(), 3) if len(s) else np.nan, note=""))
RAIL_ORDER = pd.DataFrame(rr).sort_values("sep_rel_m", na_position="last")
disp(RAIL_ORDER, title="5a &middot; SIGNAL 3 — rails, with coverage so an absent rail says so "
                       "instead of exploding", save="v5_rail_order")

# ── SIGNAL 7 · fewer or smaller. v4 answered this cleanly; re-stated on dd
TICKET = pv(CUR, ["n_out", "avg_ticket_out", "amt_out"]).round(3)
disp(TICKET.reset_index(), title="5b &middot; SIGNAL 7 — count vs ticket, on dd",
     n=EVENT_PRE+EVENT_POST+1, save="v5_ticket")
DECOMP = None
_t = TICKET.dropna()
if not _t.empty and ("n_out", "attriter") in _t.columns:
    DECOMP = pd.DataFrame({"rel_m": _t.index,
                           "count_dd":  _t[("n_out", "attriter")].values,
                           "ticket_dd": _t[("avg_ticket_out", "attriter")].values,
                           "amount_dd": _t[("amt_out", "attriter")].values})
    DECOMP["driver"] = np.where(DECOMP.count_dd < DECOMP.ticket_dd, "FEWER payments", "SMALLER payments")
    disp(DECOMP, title="5c &middot; Which drives the fall", n=EVENT_PRE+EVENT_POST+1, save="v5_ticket_decomp")

# ── SIGNAL 4 · net flow sign. v4's strongest clean trend: 0.474 -> 0.702
#    for attriters against a flat 0.45 for stayers.
NETSIGN = CUR[CUR.feature == "net_flow"].pivot_table(index="rel_m", columns="cohort", values="rate_neg").round(3)
disp(NETSIGN.reset_index(), title="5d &middot; SIGNAL 4 — share running negative net flow",
     n=EVENT_PRE+EVENT_POST+1, save="v5_netflow_sign")

# ── stacking, now on dd ───────────────────────────────────────────────
flag = (F.when(F.col("feature").isin(*NEW_ENTITY) & F.col("value").isNotNull(), (F.col("value") <= 0).cast("int"))
         .when((F.col("feature") == "net_flow") & F.col("value").isNotNull(), (F.col("value") < 0).cast("int"))
         .when(F.col("dd").isNotNull(), (F.col("dd") < SCORE_THRESH).cast("int")).otherwise(None))
sc = (ES.filter(F.col("feature").isin(*RULE.keys())).withColumn("fired", flag)
      .groupBy("cust_pwr_id", "cohort", "rel_m")
      .agg(F.sum("fired").alias("k"), F.count("fired").alias("m")).filter("m >= 6"))
p = PREV["A_full_exit"]
raw = (sc.filter(F.col("rel_m").between(-EVENT_PRE, -1)).groupBy("rel_m", "cohort")
       .agg(F.count("*").alias("n"), *[F.sum((F.col("k") >= i).cast("int")).alias(f"ge{i}") for i in range(1, 10)])
       ).toPandas()
rows = []
for rm, g in raw.groupby("rel_m"):
    a, s = g[g.cohort == "attriter"], g[g.cohort == "stayer"]
    if a.empty or s.empty: continue
    a, s = a.iloc[0], s.iloc[0]
    if a.n < MIN_CELL_N or s.n < MIN_CELL_N: continue
    for i in range(1, 10):
        rec, fpr = a[f"ge{i}"]/a.n, s[f"ge{i}"]/s.n
        den = p*rec + (1-p)*fpr; pr = (p*rec/den) if den > 0 else np.nan
        rows.append(dict(rel_m=int(rm), min_signals=i, recall=rec, fpr=fpr, precision=pr,
                         lift=pr/p if pr == pr else np.nan))
SCORE = pd.DataFrame(rows); SCORE.to_csv(OUT_DIR / "v5_score.csv", index=False)
disp(SCORE.pivot_table(index="min_signals", columns="rel_m", values="lift").round(1).reset_index(),
     title=f"5e &middot; LIFT by how many of the twelve fire (dd < {SCORE_THRESH})", n=12, save="v5_score_lift")
disp(SCORE.pivot_table(index="min_signals", columns="rel_m", values="recall").round(3).reset_index(),
     title="5f &middot; Recall — the trade against 5e", n=12, save="v5_score_recall")

# ── single vs combined, WITHIN each month and at matched recall ───────
# v5 compared "best single 30.1x" against "best combined 16.3x" and
# called it a loss. The 30.1x was at rel_m -1, where everything works;
# the 16.3x was at rel_m -12. Different months, different questions.
# Early warning lives in the far months, so the comparison is made month
# by month - and the single side is restricted to features that cleared
# the coverage floor, because the combined score is not carried by a
# 2.6%-coverage rail either.
_op = OP["A_full_exit"]
_op = _op[_op.feature.isin(ELIG)] if len(ELIG) else _op

def _best(g, col="lift", rec=None):
    h = g if rec is None else g[g.recall >= rec]
    if h.empty or h[col].isna().all(): return np.nan, None
    i = h[col].idxmax()
    return h.loc[i, col], h.loc[i, "feature" if "feature" in h.columns else "min_signals"]

rows = []
for rm in sorted(set(_op.rel_m) & set(SCORE.rel_m)):
    gs, gc = _op[_op.rel_m == rm], SCORE[SCORE.rel_m == rm]
    s_l, s_f = _best(gs); c_l, c_k = _best(gc)
    s_lr, s_fr = _best(gs, rec=CMP_RECALL); c_lr, c_kr = _best(gc, rec=CMP_RECALL)
    rows.append(dict(rel_m=int(rm),
                     best_single_lift=round(s_l, 1) if s_l == s_l else np.nan,
                     best_single_feature=s_f,
                     best_combined_lift=round(c_l, 1) if c_l == c_l else np.nan,
                     best_combined_k=c_k,
                     winner=("combined" if (c_l == c_l and s_l == s_l and c_l > s_l) else
                             "single" if s_l == s_l else None),
                     single_at_recall=round(s_lr, 1) if s_lr == s_lr else np.nan,
                     combined_at_recall=round(c_lr, 1) if c_lr == c_lr else np.nan,
                     winner_at_recall=("combined" if (c_lr == c_lr and s_lr == s_lr and c_lr > s_lr)
                                       else "single" if s_lr == s_lr else None)))
VS = pd.DataFrame(rows).sort_values("rel_m")
VS.to_csv(OUT_DIR / "v6_single_vs_combined.csv", index=False)
disp(VS, title=f"5g &middot; Single vs combined, WITHIN each month (single side restricted to "
               f"features clearing {MIN_COVERAGE:.0%} coverage)", n=20, save="v6_single_vs_combined")

_far = VS[VS.rel_m <= -6]
_near = VS[VS.rel_m >= -3]
kv({"months where combining wins (any recall)": int((VS.winner == "combined").sum()),
    "  ...of": len(VS),
    "months where combining wins at recall >= %.0f%%" % (CMP_RECALL*100):
        int((VS.winner_at_recall == "combined").sum()),
    "FAR window (rel_m <= -6): combined wins": int((_far.winner == "combined").sum()),
    "  ...of": len(_far),
    "NEAR window (rel_m >= -3): combined wins": int((_near.winner == "combined").sum()),
    "  ...of": len(_near),
    "v5 said (wrong comparison)": "no - 30.1x at rel_m -1 vs 16.3x at rel_m -12"},
   title="5h &middot; Where does stacking actually help?", save="v6_stack_verdict")

_fw = int((_far.winner == "combined").sum())
note("STACK", "Do the twelve stack, compared within each month?",
     f"combined wins in {int((VS.winner=='combined').sum())} of {len(VS)} months; "
     f"{_fw} of {len(_far)} in the far window (rel_m <= -6)",
     "v5 compared a rel_m -1 single lift against a rel_m -12 combined lift and called it a loss. "
     "Early warning lives in the far months, and that is where a combined rule has to earn its "
     "keep. If it wins there and loses near the event, that is still the right answer for a "
     "queue that is meant to give warning rather than confirm a departure.")

## 6 · HTML report

In [ ]:
# =====================================================================
# 7 · THE TWELVE-SIGNAL HTML REPORT
# =====================================================================
CSS = """
:root{--ink:#16181D;--mut:#6B7280;--line:#E5E7EB;--acc:#C1440E;--acc2:#4A6FA5;--bg:#FCFCFD}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.65 "IBM Plex Sans",-apple-system,Segoe UI,sans-serif}
.wrap{max-width:1080px;margin:0 auto;padding:48px 32px 96px}
h1{font-size:30px;font-weight:600;margin:0 0 6px;letter-spacing:-.02em}
h2{font-size:20px;font-weight:600;margin:0 0 4px;letter-spacing:-.01em}
h3{font-size:14px;margin:18px 0 4px}
.sub{color:var(--mut);font-size:14px;margin:0 0 30px}
.sig{border:1px solid var(--line);border-radius:10px;background:#fff;padding:24px 26px;margin:0 0 20px}
.sig-h{display:flex;align-items:baseline;gap:12px;border-bottom:1px solid var(--line);padding-bottom:12px;margin-bottom:16px}
.num{font:600 13px "IBM Plex Mono",monospace;color:#fff;background:var(--acc);border-radius:4px;padding:3px 8px;flex:none}
.feat{font:12px "IBM Plex Mono",monospace;color:var(--mut);margin-left:auto}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:14px 0}
.card{border:1px solid var(--line);border-radius:8px;padding:11px 13px;background:var(--bg)}
.card .k{font-size:11px;color:var(--mut);text-transform:uppercase;letter-spacing:.05em}
.card .v{font:600 21px "IBM Plex Mono",monospace;margin-top:3px}.card .v.hi{color:var(--acc)}
table.t{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0}
table.t th{text-align:left;font-weight:600;color:var(--mut);font-size:11px;text-transform:uppercase;
 letter-spacing:.04em;border-bottom:1px solid var(--line);padding:7px 10px}
table.t td{padding:6px 10px;border-bottom:1px solid #F3F4F6;font-variant-numeric:tabular-nums}
table.t tr:hover td{background:#FAFAFB}
.read{background:#FFF8F4;border-left:3px solid var(--acc);padding:11px 15px;margin:14px 0;font-size:14px}
.warn{background:#F5F7FA;border-left:3px solid var(--acc2);padding:11px 15px;margin:14px 0;font-size:14px}
.charts{display:grid;grid-template-columns:repeat(auto-fit,minmax(330px,1fr));gap:18px;margin:16px 0}
footer{color:var(--mut);font-size:12px;border-top:1px solid var(--line);padding-top:18px;margin-top:40px}
"""
def _cards(it):
    return "<div class='grid'>" + "".join(
        f"<div class='card'><div class='k'>{_html.escape(k)}</div><div class='v{' hi' if h else ''}'>{v}</div></div>"
        for k, v, h in it) + "</div>"
def _f(v, s="{:.3f}"):
    return "&mdash;" if v is None or (isinstance(v, float) and pd.isna(v)) else s.format(v)

def signal_block(s):
    f = s["feature"]
    row = BEST["A_full_exit"][BEST["A_full_exit"].feature == f]
    row = row.iloc[0] if len(row) else None
    sp = LEAD[LEAD.feature == f]; sp = sp.iloc[0] if len(sp) else None
    col, _ = col_for(f)
    cur = (CUR[CUR.feature == f].pivot_table(index="rel_m", columns="cohort", values=col))
    cur = cur.reindex(columns=[c for c in ("attriter", "stayer") if c in cur.columns])
    ylab = {"rate_any": "share with any", "rate_neg": "share negative"}.get(col, "own change vs peers' change")
    chart = svg_lines(cur, list(cur.columns), title=ylab, yzero=(col != "med_dd"),
                      hline=1.0 if col == "med_dd" else None)
    opf = OP["A_full_exit"]; opf = opf[opf.feature == f]
    lb = opf.groupby("rel_m").lift.max().sort_index() if len(opf) else pd.Series(dtype=float)
    chart2 = svg_bars([str(int(i)) for i in lb.index], list(lb.values), title="best lift, by month") if len(lb) else ""
    cov = None if sp is None else sp.coverage
    elig = bool(sp.eligible) if sp is not None else False
    cards = [("separates at", f"{int(sp.sep_rel_m)} mo" if sp is not None and pd.notna(sp.sep_rel_m) else "&mdash;", elig),
             ("coverage", _f(cov, "{:.1%}"), False),
             ("best lift", _f(None if row is None else row.best_lift, "{:.1f}&times;"), True),
             ("at month", _f(None if row is None else row.best_at_rel_m, "{:.0f}"), False),
             ("recall there", _f(None if row is None else row.best_recall), False)]
    extra = ""
    if not elig:
        extra += (f"<div class='warn'><b>Too sparse to rank.</b> A dd exists only for customers "
                  f"who used this in both windows, and coverage here is {_f(cov,'{:.1%}')} against a "
                  f"{MIN_COVERAGE:.0%} floor. The curve is a median over a self-selected subset, so it "
                  f"is shown but excluded from the ordering claim.</div>")
    if s["n"] == 3 and len(RAIL_ORDER): extra += "<h3>Rails, in the order they go</h3>" + tbl(RAIL_ORDER)
    if s["n"] == 7 and DECOMP is not None:
        extra += ("<h3>Fewer, or smaller?</h3>" + tbl(DECOMP.round(3)) +
                  "<div class='read'>Where <b>count_dd</b> sits below <b>ticket_dd</b> they make "
                  "<b>fewer</b> payments of much the same size &mdash; the relationship is being wound "
                  "down, not the business shrinking.</div>")
    if s["n"] == 4 and len(NETSIGN): extra += "<h3>Share running negative</h3>" + tbl(NETSIGN.reset_index().round(3))
    if s["n"] == 9:
        extra += ("<h3>The incumbent, measured</h3>" + tbl(BENCH) +
                  "<div class='warn'>Everything above is worth deploying only to the extent it beats "
                  "this &mdash; and the 30% rule also fired on 43,143 customers who never left.</div>")
    return (f"<section class='sig'><div class='sig-h'><span class='num'>{s['n']:02d}</span>"
            f"<h2>{_html.escape(s['name'])}</h2><span class='feat'>{_html.escape(f)}</span></div>"
            + _cards(cards) + f"<div class='charts'><div>{chart}</div><div>{chart2}</div></div>" + extra + "</section>")

_mk = MARKER.toPandas() if hasattr(MARKER, "toPandas") else MARKER
_sc = SCORE.pivot_table(index="min_signals", columns="rel_m", values="lift").round(1).reset_index() if len(SCORE) else pd.DataFrame()

doc = f"""<!doctype html><html><head><meta charset="utf-8">
<title>PKG — Twelve early-warning signals</title><style>{CSS}</style></head><body><div class="wrap">
<h1>Twelve early-warning signals</h1>
<p class="sub">Payment Knowledge Graph &middot; PNC Treasury Management &middot; Data Science &middot;
generated {dt.date.today().isoformat()} &middot; <b>A_full_exit</b>,
{B['A_full_exit']['n_att']:,} attriters vs {B['A_full_exit']['n_sta']:,} stayers</p>

<div class="warn"><b>How to read this.</b> Charts run in months before the customer leaves; 0 is the exit
month. Lines are <b>difference-in-differences</b>: the customer's own change over the last six months,
divided by the same change across its size peers. <b>1.0 is normal</b> &mdash; below 1.0 means falling
faster than peers. <b>Lift</b> is how much more likely a flagged customer is to leave than a random one;
the base rate is {PREV['A_full_exit']:.2%} a month, so 20&times; lift is roughly 1 in 5 rather than 1 in 110.</div>

<div class="read"><b>Why difference-in-differences.</b> Nothing before 2024 is trusted and none can be
added, so a twelve-month self-baseline would discard about a quarter of the attriters permanently. Peer
levels alone were not the answer either &mdash; a level ratio sits at 1.0 across the whole population, so
any threshold below 1.0 flags roughly half of everyone. Measuring each customer's <i>change</i> against
its peers' <i>change</i> keeps what both were good for. It needs six months of history rather than twelve,
and it is computable in production every month without knowing the exit date in advance.</div>

<h2 style="margin:34px 0 8px">Ordering: what moves first</h2>
{tbl(LEAD)}
<div class="read"><b>Coverage matters more than it looks.</b> A dd exists only for customers who used
that feature in both windows. Features below the {MIN_COVERAGE:.0%} floor are shown but cannot carry an
ordering claim &mdash; their medians are computed over a self-selected subset of users, which is how a
rail with 2.6% coverage came to top the previous headline.</div>

{''.join(signal_block(s) for s in SIGNALS)}

<section class="sig"><div class="sig-h"><span class="num">&#9679;</span>
<h2>Standing marker: they were different all along</h2><span class="feat">level vs peers at rel_m &minus;12</span></div>
{tbl(_mk)}
<div class="read">A full year before they leave, attriters are already materially lighter payment users
than their size peers. This has no onset date, so it is not an early warning &mdash; it is a
<b>segmentation marker</b>, useful for deciding who to watch rather than when to act.</div></section>

<section class="sig"><div class="sig-h"><span class="num">&Sigma;</span>
<h2>Do they stack?</h2><span class="feat">unweighted count, dd &lt; {SCORE_THRESH}</span></div>
<h3>Single vs combined, within each month</h3>
{tbl(VS)}
<div class="read">The comparison has to be made <b>within</b> a month. A single feature at rel_m
&minus;1 and a combined rule at &minus;12 answer different questions &mdash; one confirms a departure,
the other gives warning. The single side is restricted to features clearing the
{MIN_COVERAGE:.0%} coverage floor, because the combined score is not carried by a sparse rail either.
<b>The far window is the one that matters</b>: a queue meant to give warning has to earn its keep at
&minus;6 and beyond, not at &minus;1.</div>
<h3>Lift by how many fire</h3>
{tbl(_sc)}
<div class="read">A plain count, no fitting, auditable line by line.</div></section>

<footer>Generated by <code>pkg_attrition_eda_v6.ipynb</code>. Tables in <code>{OUT_DIR}</code>.
Internal &mdash; PNC Treasury Management, Data Science.</footer>
</div></body></html>"""
(OUT_DIR / HTML_NAME).write_text(doc, encoding="utf-8")
print("wrote", OUT_DIR / HTML_NAME, f"({len(doc):,} bytes)")
display(HTML(f"<a href='{HTML_NAME}' target='_blank' style='font:600 14px IBM Plex Sans'>open {HTML_NAME}</a>"))
disp(pd.DataFrame(FINDINGS), title="6 &middot; v6 findings", n=40, save="FINDINGS_v6")
print("\nLocal:", OUT_DIR)
for f in sorted(OUT_DIR.glob("*")): print("  ", f.name)

---

## After this run

1. **§4a2 lists what got excluded.** If a signal you care about is there, it is not that the
   signal is weak — it is that too few customers use that rail for a median to mean anything.
   Report it as a within-segment finding or not at all.
2. **§5h is the go/no-go for modelling.** If combining wins in the far window, build the
   discrete-time hazard model. If it wins only near the event, a combined rule confirms
   departures rather than predicting them, and the effort belongs elsewhere.
3. **The 2-month honest lead is still worth shipping** — it beats the incumbent's median 2
   months at far better precision, and covers the 36% the 30% rule never reaches at all.
4. **Then counterparty data.** Everything here is the PNC-visible slice; `PAYS_CPTY` and
   `CptyFinEntity` are what would let "banks they pay drop away" be measured properly.